# Библиотека LightFM

## 1. Описание библиотеки

**LightFM** – это библиотека Python, разработанная специально для создания рекомендательных систем. Она предоставляет инструменты для обучения моделей, которые могут предсказывать предпочтения пользователей на основе их предыдущих взаимодействий с элементами системы (например, покупки товаров, просмотры фильмов и т.п.). Основная цель библиотеки – сделать процесс разработки рекомендательных систем простым и эффективным, предоставляя удобные интерфейсы для работы с данными и моделями.

**Основные особенности LightFM:**
1. ***Интеграция факторов:***
    * LightFM позволяет использовать различные типы данных при обучении модели. Например, вы можете учитывать не только информацию о том, какие товары покупал пользователь, но также демографические данные пользователя, характеристики товара и другие факторы.
2. ***Методы оптимизации:***
    * Библиотека поддерживает несколько методов оптимизации, включая SGD (Stochastic Gradient Descent), Adam и Adagrad. Это позволяет гибко настраивать процесс обучения модели под конкретные задачи.
3. ***Поддержка гибридных рекомендаций:***
    * LightFM может комбинировать разные подходы к созданию рекомендаций, такие как коллаборативная фильтрация и контентная фильтрация. Это помогает улучшить качество рекомендаций за счет использования различных источников информации.
4. ***Простота использования:***
    * Интерфейсы библиотеки интуитивны и просты в использовании. Вы можете быстро начать работу с ней даже без глубокого понимания математических основ рекомендательных систем.
5. ***Высокая производительность:***
    * LightFM оптимизирована для работы с большими объемами данных. Она использует Cython для ускорения вычислений, что делает ее подходящей для задач реального времени.
6. ***Визуализация результатов:***
    * Библиотека предоставляет функции для визуализации результатов обучения модели, что облегчает анализ качества рекомендаций.


**Установка**

Для установки LightFM вам потребуется Python версии 3.6 или выше. Вы можете установить библиотеку через pip:
```
pip install lightfm
```
```
conda install -c conda-forge lightfm
```

**LightFM** – это мощная и удобная библиотека для создания рекомендательных систем. Она предлагает широкий спектр возможностей для настройки и оптимизации моделей, а также обеспечивает высокую производительность при работе с большими наборами данных. Если вы работаете над проектом, связанным с рекомендациями, LightFM станет отличным выбором благодаря своей простоте и эффективности.

**Документация и ресурсы**
* [Официальный сайт](https://making.lyst.com/lightfm/docs/home.html)
* [Страница на PyPi](https://pypi.org/project/lightfm/)
* [Репозиторий на GitHub](https://github.com/lyst/lightfm)

#### **Перекрестная проверка / Cross-validation**

Функции разделения набора данных.
```
lightfm.cross_validation.random_train_test_split(interactions, test_percentage=0.2, random_state=None)[source]
```
Случайное разделение взаимодействий между обучением и тестированием.

Эта функция берет набор взаимодействий и разбивает его на два непересекающихся набора, обучающий набор и тестовый набор. Обратите внимание, что не прилагается никаких усилий для того, ***чтобы убедиться, что все элементы и пользователи, взаимодействующие в тестовом наборе***, также взаимодействуют в обучающем наборе; это может привести к частичной проблеме холодного запуска в тестовом наборе. Чтобы разделить матрицу sample_weight по тем же строкам, передайте ее в эту функцию с тем же начальным значением random_state, которое использовалось для разделения взаимодействий.

**Parameters**
* interactions (разреженная матрица scipy, содержащая взаимодействия) – Взаимодействия для разделения.
* test_percentage (float, optional) – Доля взаимодействий, которые нужно поместить в тестовый набор.
* random_state (int or numpy.random.RandomState, optional) – Случайное начальное значение, используемое для инициализации numpy.random. Генератор чисел RandomState. Принимает экземпляр numpy.random.RandomState для обратной совместимости.

**Returns:** (train, test) – scipy.sparse.COOMatrix) A tuple of (train data, test data)

**Return type:** (scipy.sparse.COOMatrix,)


In [1]:
# загружаем необходимые библиотеки, методы и функции
import numpy as np  # импортируем NumPy для работы с массивами и матрицами
import pandas as pd  # импортируем Pandas для работы с таблицами данных
import time # импортируем функцию для определения времени выполнения кода


import scipy # Импортируем библиотеку для научных вычислений
from scipy.sparse import coo_matrix  # Импорт функции coo_matrix из библиотеки SciPy для создания разреженных матриц в формате COOrdinate
from scipy.sparse import csr_matrix # Импорт функции csc_matrix из библиотеки SciPy для создания разреженных матриц в формате CSC

from sklearn.feature_extraction.text import TfidfVectorizer # Импортируем класс для векторизации текстов с использованием TF-IDF
from sklearn.metrics.pairwise import cosine_similarity # Импортируем функцию для вычисления косинусной меры схожести между векторами

import optuna # библиотека подбора гиперпараметров ML моделей

from joblib import dump, load # библиотека для сохранения и загрузки моделей

from lightfm import LightFM # библиотека для построения моделей с помощью LightFM
from lightfm.evaluation import precision_at_k # метрика качества модели - точность на k предсказаниях
from lightfm.cross_validation import random_train_test_split # функция для разделения данных на тренировочную и тестовую выборки
from lightfm.data import Dataset # класс для хранения данных
from lightfm.evaluation import auc_score # метрика качества модели

# фиксируем RANDOM_SEED, для воспроизводимости кода.
RANDOM_SEED = 369

# отключаем предупреждения
import warnings 
warnings.filterwarnings("ignore")

# увеличиваем количество отображаемых столбцов и строк
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 3000)

%load_ext watermark
%watermark -n -u -v -iv -w

Last updated: Sat Jan 25 2025

Python implementation: CPython
Python version       : 3.11.11
IPython version      : 8.30.0

scipy  : 1.15.1
numpy  : 1.26.4
sklearn: 1.5.2
optuna : 4.1.0
lightfm: 1.17
joblib : 1.4.2
pandas : 2.2.3

Watermark: 2.5.0



/Users/varlaam/anaconda3/envs/FP_LightFM/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
# conda install -c conda-forge optuna
# conda install -c conda-forge lightfm
# pip install watermark
# pip install --upgrade jupyter ipywidgets
# pip install --upgrade pip setuptools

## 2. Подготовка данных

In [3]:
# cоздаем пустой DataFrame для хранения результатов
results = pd.DataFrame(columns=["Model", "Time", "train_Precision@3", "test_Precision@3"])

# функция для добавления результатов в results_lightfm
def add_results(results_lightfm, model_name, elapsed_time, train_precision, test_precision):
    """_summary_

    Args:
        results_lightfm (_type_): _description_
        model_name (_type_): _description_
        elapsed_time (_type_): _description_
        train_precision (_type_): _description_
        test_precision (_type_): _description_

    Returns:
        _type_: _description_
    """
    new_row = pd.Series({
        "Model": model_name,
        "Time": elapsed_time,
        "train_Precision@3": train_precision,
        "test_Precision@3": test_precision
    })
    results = pd.concat([results_lightfm, new_row.to_frame().T], ignore_index=True)
    return results  # Возвращаем обновленный датафрейм

In [4]:
# Вводим переменные
data_name = "light_count"

In [5]:
# Загружаем данные c событиями
data = pd.read_csv(f"/Users/varlaam/Desktop/Data Science/1. SkillFactory/1. Курс_Profession Data Science/14. Трек ML-инженер/10. Дипломный проект/data/data_{data_name}.csv", sep=",")

In [6]:
# Загружаем и преобразуем данные со ствойствами товаров
pr1 = pd.read_csv("data/item_properties_part1.csv", sep=",") # данные со свойствами товаров часть-1
pr2 = pd.read_csv("data/item_properties_part2.csv", sep=",") # данные со свойствами товаров часть-2
pr = pd.concat([pr1, pr2])
pr = pr.drop(columns=["timestamp"], axis=1)
# Создаем dataframe группируя данные по itemid и объединяя все значения свойства
property_groupby = pr.groupby("itemid")["value"].apply(lambda x: ' '.join(x)).to_frame()
# Очищаем признак "value" от дубликатов и преобразуем в строковый тип
property_groupby["value"] = property_groupby["value"].apply(lambda x: str(set(x.split(" "))))

# Проводим векторизацию свойст товаров и создаем новый DataFrame.
tfidfvec = TfidfVectorizer(min_df=5000, max_df=0.7)
vectorized_data = tfidfvec.fit_transform(property_groupby["value"])

property_vectorized = pd.DataFrame(vectorized_data.toarray(),
                       columns=tfidfvec.get_feature_names_out())
property_vectorized.index = property_groupby.index
property_vectorized = property_vectorized.reset_index()
#display(property_vectorized.head(3))

In [7]:
# Формируем список пересекающихся товаров в данных
iteid_data_list = data["itemid"].unique().tolist()
itemid_property_list = property_vectorized["itemid"].index.to_list()
itemid_intersection = list(set(iteid_data_list) & set(itemid_property_list))
# Фильтруем данные, оставляя только наблюдения по пересекающимся товарам для построения гибридной системы
data_intersection = data[data["itemid"].isin(itemid_intersection)]
property_intersection = property_vectorized[property_vectorized["itemid"].isin(itemid_intersection)]

In [8]:
# Создание объекта Dataset для подготовки данных
dataset = Dataset()

# Формируем список названий признаков в данных свойства
item_features_list = property_intersection.columns[1:]
# Формируем список уникальных значений юзеров и товаров
users = data_intersection["visitorid"].unique().tolist()
items = data_intersection["itemid"].unique().tolist()

# Подготовка данных для fit метода
dataset.fit(users=users, items=items, item_features=item_features_list)

# Преобразуем DataFrame в формат (item_id, [list of feature names])
property_transform = [
    (row["itemid"], [label for label in item_features_list if row[label] != 0])
    for _, row in property_intersection.iterrows()
]
# Формируем матрицу свойст товаров
item_features = dataset.build_item_features(property_transform)

In [9]:
df = data_intersection.copy()
# Нормализуем visitorid и itemid
min_visitorid = df['visitorid'].min()
max_visitorid = df['visitorid'].max()
min_itemid = df['itemid'].min()
max_itemid = df['itemid'].max()

normalized_df = df.copy()
normalized_df['visitorid'] -= min_visitorid
normalized_df['itemid'] -= min_itemid

# Уникальные пользователи и элементы
users = sorted(normalized_df['visitorid'].unique())
items = sorted(normalized_df['itemid'].unique())

# Создаем словарь для маппинга id на индексы
user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {i: j for j, i in enumerate(items)}

# Преобразуем значения в индексы
row = normalized_df['visitorid'].map(user_to_idx)
col = normalized_df['itemid'].map(item_to_idx)
values = normalized_df['like']

# Создаем разреженную матрицу
matrix = csr_matrix((values, (row, col)), shape=(len(users), len(items)))

# Отключаем проверку на пересечение данных
LightFM._check_test_train_intersections = lambda *args: None

# Формируем тренировочную и тестовую выборки
train, test = random_train_test_split(matrix, test_percentage=0.2, random_state=RANDOM_SEED)

# Оценим размеры обучающей и тестовой выборки
print(f'Размер обучающей выборки: {train.shape}')
print(f'Размер тестовой выборки: {test.shape}')
print(f'Размер выборки со свойствами товаров : {item_features.shape}')

Размер обучающей выборки: (58173, 85311)
Размер тестовой выборки: (58173, 85311)
Размер выборки со свойствами товаров : (85311, 85737)


## 3. Построение модели на базе "collaborative filtering" LightFM

**LightFM - гибридная модель рекомендаций на основе латентных представлений.**

Эта модель учится создавать представления (латентные представления в многомерном пространстве) для пользователей и элементов таким образом, чтобы зашифровать предпочтения пользователей по отношению к элементам. Умножая эти представления вместе, модель генерирует оценки для каждого элемента для заданного пользователя, и элементы с высокими оценками считаются более интересными для пользователя.

Представления пользователей и элементов выражаются через представления их характеристик: для каждой характеристики строится представление, а затем эти характеристики суммируются для формирования представлений пользователей и элементов. Например, если фильм «Волшебник страны Оз» характеризуется следующими характеристиками: «музыкально-фэнтезийный», «Джуди Гарланд» и «волшебник страны Оз», то его представление формируется путем суммирования представлений этих характеристик.

Представления изучаются с использованием стохастического градиента.

**Доступны четыре функции потерь - loss:**
1. logistic: Полезна, когда присутствуют как положительные (+1), так и отрицательные (-1) взаимодействия.
2. BPR (Bayesian Personalised Ranking): Байесианская персонализованная ранжировка. Потеря, которая максимизирует разницу в предсказаниях между положительным примером и случайно выбранным отрицательным примером. Полезна, когда имеются только положительные взаимодействия, и цель состоит в оптимизации ROC AUC.
3. WARP (Weighted Approximate-Rank Pairwise): Весовая аппроксимационная ранговая потеря. Максимизирует ранг положительных примеров путём многократного отбора отрицательных примеров до нахождения нарушения ранга. Полезна, когда есть только положительные взаимодействия, и требуется оптимизация списка рекомендаций (precision@k).
4. k-OS WARP: Потеря k-того порядка статистики. Модификация WARP, использующая k-й положительный пример для любых взаимодействий пользователя в качестве основы для обновления.

**Доступны два алгоритма скорости обучения:**
1. adagrad: Алгоритм AdaGrad - это адаптивный градиентный спуск, разработанный в 2011 году. Он является модификацией классического стохастического градиентного спуска, где для каждого параметра устанавливается своя скорость обучения. Это ускоряет обучение для параметров, которые получают малое количество изменений, и замедляет обучение для параметров, которые меняются реже. Алгоритм использует скользующую среднюю величину градиента для каждого параметра, чтобы настроить скорость обучения. 
2. adadelta: AdaDelta — это алгоритм адаптации, который использует скользящую среднюю и дисперсию для регулировки скорости обучения. AdaDelta динамически настраивает шаг размера на основе средней величины и дисперсией градиента, что позволяет быстрее учиться, сохраняя стабильность. В отличие от AdaGrad, AdaDelta дополнительно учитывает историю градиентов, что делает его более устойчивым к изменениям в данных.

**Примечания**

Представления пользователей и элементов выражаются через представления их характеристик. Если не предоставлены матрицы характеристик для методов lightfm.LightFM.fit() или lightfm.LightFM.predict(), подразумевается, что они являются единичными матрицами: то есть, каждый пользователь и элемент характеризуются одной характеристикой, уникальной для этого пользователя (или элемента). В таком случае LightFM сводится к традиционному методу совместной факторизации матриц.

Для включения характеристик существуют две стратегии:
* Характеризация каждого пользователя/элемента только его характеристиками.
* Характеризация каждого пользователя/элемента его характеристиками и единичной матрицей, которая захватывает взаимодействия между пользователями и элементами непосредственно.


1. При использовании только характеристик, матрица характеристик должна иметь форму (количество пользователей/элементов × количество характеристик). Для построения таких матриц рекомендуется использовать методы класса lightfm.data.Dataset и установить <user/item>_identity_features на False. Представление для пользователя i формируется путем нахождения ненулевых весов в i-й строке матрицы характеристик и сложения соответствующих представлений характеристик. Например, если пользователь 10 имеет вес 1 в пятой колонке матрицы характеристик и вес 3 в двадцатой колонке, его представление будет получено путем сложения представлений этих двух характеристик (при этом представление третьей характеристики умножается на 3).
>Замечание: эта стратегия может приводить к менее выразительной модели, так как индивидуальные характеристики пользователя не учитываются. Чтобы преодолеть это, следуйте стратегии 2 и включайте индивидуальные характеристики в матрицу характеристик.

2. Чтобы включить характеристики наряду с взаимодействиями между пользователями и элементами, матрица характеристик должна включать единичную матрицу. Тогда результирующая матрица характеристик будет иметь форму (количество пользователей/элементов × (количество пользователей/элементов + количество характеристик)). Это поведение контролируется аргументом <user/item>_identity_features, установленным по умолчанию на True. В этом случае модель будет учитывать как взаимодействия, так и индивидуальные характеристики.
>Эти стратегии помогают управлять степенью выразительности модели и балансировать между учетом индивидуальных особенностей и глобальных взаимодействий.

In [10]:
losses = ["bpr", "logistic", "warp", "warp-kos"]

for loss in losses:
    # фиксируем время начала работы
    start_time = time.time()
    # формируем параметры модели и обучаем её
    model = LightFM(learning_rate=1e-4, 
                    loss=loss,
                    )
    model.fit(train, epochs=100)

    # расчитываем метрику precision@k
    train_precision = precision_at_k(model, train, k=3).mean()
    test_precision = precision_at_k(model, test, k=3).mean()

    # фиксируем время окончания работы
    end_time = time.time()
    elapsed_time = end_time - start_time

    # # выводим результаты метрик
    # print("Precision@3: train %.4f, test %.4f." % (train_precision, test_precision))
    # print()

    # # Вариан 1 - Выбираем случайного пользователя
    # random_normalized_visitorid = np.random.choice(users)

    # # Вариант 2 - Вводим оригинальный visitorid
    # # original_visitorid = 391657
    # # random_normalized_visitorid = original_visitorid - min_visitorid # приводим его к нормализованному виду

    # userid = user_to_idx[random_normalized_visitorid]

    # # получаем рекомендации для пользователя
    # n_users, n_items = matrix.shape
    # scores = model.predict(userid, np.arange(n_items), num_threads=8)
    # top_items = np.argsort(-scores)[:3]  # берем топ-3 элемента

    # # преобразуем обратно в оригинальные itemid
    # original_item_ids = [items[idx] + min_itemid for idx in top_items]

    # # преобразуем обратно в оригинальный visitorid
    # original_visitorid = random_normalized_visitorid + min_visitorid

    # # используем pandas для отображения результатов в виде таблицы
    # rec_user = pd.DataFrame({
    #     "visitorid": original_visitorid,
    #     "itemid": original_item_ids,
    #     "score": scores[top_items],
    #     "already_liked": np.in1d(top_items, matrix[userid].indices.ravel())  # преобразуем в одномерный массив
    # })

    # print(rec_user)
    # print()

    # set1 = set(df[df["visitorid"] == original_visitorid]["itemid"].unique().tolist())
    # set2 = set(original_item_ids)
    # print(f'Соответствие itemid между рекомендациями и фактом: {set1 & set2}')
    # print()

    # print(rec_user["already_liked"].value_counts())
    # print()

    # сохраняем результаты
    results = add_results(results, f"LightFM_{data_name}_{loss}", elapsed_time, train_precision, test_precision)
print(results)


                          Model        Time train_Precision@3 test_Precision@3
0       LightFM_light_count_bpr  365.955931           0.00017         0.000018
1  LightFM_light_count_logistic  344.086594          0.004212         0.001503
2      LightFM_light_count_warp  365.557477          0.004288         0.001439
3  LightFM_light_count_warp-kos   384.49376          0.002981         0.000947


In [11]:
# Формируем параметры модели, обучаем и выводим результаты
model = LightFM(learning_rate=1e-4, 
                loss="warp",
                )
model.fit(train, epochs=200)

# расчитываем метрику precision@k
train_precision = precision_at_k(model, train, k=3).mean()
test_precision = precision_at_k(model, test, k=3).mean()

print("Precision@3: train %.4f, test %.4f." % (train_precision, test_precision))

Precision@3: train 0.0048, test 0.0016.


In [12]:
# Оценим влияние смещения на результат
biases_ = 0    
model.item_biases *= biases_
# расчитываем метрику precision@k
train_precision = precision_at_k(model, train, k=3).mean()
test_precision = precision_at_k(model, test, k=3).mean()

print(f'Precision@3 со смещением {biases_}: {train_precision:.4f}, {test_precision:.4f}')


Precision@3 со смещением 0: 0.0001, 0.0000


In [13]:
# # Формируем рекомендации для покупателей
# # Вариан 1 - Выбираем случайного пользователя
# random_normalized_visitorid = np.random.choice(users)

# Вариант 2 - Вводим оригинальный visitorid
original_visitorid = 890980
random_normalized_visitorid = original_visitorid - min_visitorid # приводим его к нормализованному виду

userid = user_to_idx[random_normalized_visitorid]

# получаем рекомендации для пользователя
n_users, n_items = matrix.shape
scores = model.predict(userid, np.arange(n_items), num_threads=8)
top_items = np.argsort(-scores)[:3]  # берем топ-3 элемента

# преобразуем обратно в оригинальные itemid
original_item_ids = [items[idx] + min_itemid for idx in top_items]

# преобразуем обратно в оригинальный visitorid
original_visitorid = random_normalized_visitorid + min_visitorid

# используем pandas для отображения результатов в виде таблицы
rec_user = pd.DataFrame({
    "visitorid": original_visitorid,
    "itemid": original_item_ids,
    "score": scores[top_items],
    "already_liked": np.in1d(top_items, matrix[userid].indices.ravel())  # преобразуем в одномерный массив
})

print(rec_user)

   visitorid  itemid     score  already_liked
0     890980  121745 -0.042768          False
1     890980  164547 -0.043197          False
2     890980  284498 -0.043349          False


## 4. Построение моделей на базе "hybrid model" LightFM

Попробуем улучшить результат метрики Precission@3, используя возможности гибридной модели LightFM c использованием свойств товаров.

In [14]:
# Устанавливаем базовые параметры
NUM_COMPONENTS = 50
NUM_EPOCHS = 10
ITEM_ALPHA = 1e-4

# Фиксируем время начала работы
start_time = time.time()
    
# Формируем параметры модели
model_hybrid = LightFM(loss='warp',
                item_alpha=ITEM_ALPHA,
                no_components=NUM_COMPONENTS)

# Обучение гибридной модели. Обратите внимание.
# Обратите внимание, что на этот раз мы проходим по элементам в матрице элементов.
model_hybrid = model_hybrid.fit(train,
                item_features=item_features,
                epochs=NUM_EPOCHS,
                )

# Проверка размеров матрицы признаков и модели
print("Item feature matrix shape:", item_features.shape)
print("Model item embedding shape:", model_hybrid.item_embeddings.shape)

# Оценим метрику precision@k
train_precision = precision_at_k(model_hybrid, train, item_features=item_features, k=3).mean()
test_precision = precision_at_k(model_hybrid, test, item_features=item_features, k=3).mean()

# Фиксируем время окончания работы
end_time = time.time()
elapsed_time = end_time - start_time

# Записываем результаты в файл
results = add_results(results, f"LightFM_light_count_hybrid", elapsed_time, train_precision, test_precision)
    
print(f'Precision@3: {train_precision:.4f}, {test_precision:.4f}')
print()
print(results)


Item feature matrix shape: (85311, 85737)
Model item embedding shape: (85737, 50)
Precision@3: 0.0272, 0.0009

                          Model         Time train_Precision@3  \
0       LightFM_light_count_bpr   365.955931           0.00017   
1  LightFM_light_count_logistic   344.086594          0.004212   
2      LightFM_light_count_warp   365.557477          0.004288   
3  LightFM_light_count_warp-kos    384.49376          0.002981   
4                LightFM_hybrid  2046.534966          0.027237   

  test_Precision@3  
0         0.000018  
1         0.001503  
2         0.001439  
3         0.000947  
4         0.000892  


In [15]:
# # Формируем рекомендации для покупателей
# # Вариан 1 - Выбираем случайного пользователя
# random_normalized_visitorid = np.random.choice(users)

# Вариант 2 - Вводим оригинальный visitorid
original_visitorid = 890980
random_normalized_visitorid = original_visitorid - min_visitorid # приводим его к нормализованному виду

userid = user_to_idx[random_normalized_visitorid]

# получаем рекомендации для пользователя
n_users, n_items = matrix.shape
scores = model_hybrid.predict(userid, np.arange(n_items), num_threads=8)
top_items = np.argsort(-scores)[:3]  # берем топ-3 элемента

# преобразуем обратно в оригинальные itemid
original_item_ids = [items[idx] + min_itemid for idx in top_items]

# преобразуем обратно в оригинальный visitorid
original_visitorid = random_normalized_visitorid + min_visitorid

# используем pandas для отображения результатов в виде таблицы
rec_user = pd.DataFrame({
    "visitorid": original_visitorid,
    "itemid": original_item_ids,
    "score": scores[top_items],
    "already_liked": np.in1d(top_items, matrix[userid].indices.ravel())  # преобразуем в одномерный массив
})

print(rec_user)

   visitorid  itemid     score  already_liked
0     890980  320130 -2.317528          False
1     890980  213834 -2.776791           True
2     890980  219512 -3.038060          False


In [ ]:
print(stop)

## 5. Подбор гиперпараметров с помощью **OPTUNA**

In [17]:
def objective(trial):
    # получаем гиперпараметры из trial
    no_components = trial.suggest_int("no_components", 10, 100)
    k = trial.suggest_int("k", 1, 10)
    n = trial.suggest_int("n", 10, 50)
    loss = trial.suggest_categorical("loss", ["warp", "bpr", "logistic", "warp-kos"])
    learning_schedule = trial.suggest_categorical("learning_schedule", ["adagrad", "adadelta"])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-6, 1e-2)
    rho = trial.suggest_uniform("rho", 0.01, 0.999)
    epsilon = trial.suggest_loguniform("epsilon", 1e-8, 1e-1)
    item_alpha = trial.suggest_loguniform("item_alpha", 1e-6, 1e-2)
    user_alpha = trial.suggest_loguniform("user_alpha", 1e-6, 1e-2)
    max_sampled = trial.suggest_int("max_sampled", 10, 100)
    random_state = RANDOM_SEED 

    # Обучаем модель с новыми параметрами
    model = LightFM(no_components=no_components,            # размерность скрытых латентных представлений (embeddings) для пользователей и элементов. Чем больше это значение, тем сложнее модель, 
                                                                # но потенциально она сможет уловить более сложные взаимосвязи между пользователями и элементами. Увеличение этого параметра может повысить точность модели, 
                                                                # но также увеличивает время тренировки.
                    k=k,                                    # параметр для обучения с выбором k-го положительного примера из n положительных примеров, взятых для каждого пользователя. 
                                                                # этот параметр используется для обучения с потерей warp-kos. Он указывает, какой положительный пример будет выбран для обучения.
                    n=n,                                    # максимальное количество положительных примеров, которые будут выбраны для каждого обновления.
                                                                # вместе с параметром k, этот параметр контролирует выбор положительных примеров для обучения. Большее значение может увеличить точность, но потребует больше времени на обучение.
                    learning_schedule=learning_schedule,    # стратегия обучения, которая управляет изменением весов модели во время обучения. Может принимать значения 'adagrad' или 'adadelta'.
                    loss=loss,                              # функция потерь, используемая для оптимизации модели. Доступны варианты: 'logistic', 'bpr', 'warp', 'warp-kos'.
                    learning_rate=learning_rate,            # начальная скорость обучения, определяющая величину изменения весов на каждом этапе обучения.
                                                                # маленькие значения делают обучение медленнее, но стабильнее, большие значения ускоряют обучение, но могут привести к переобучению.
                    rho=rho,                                # коэффициент затухания для стратегии adadelta, который контролирует влияние предыдущих градиентов на обновление весов.
                                                                # значение находится в диапазоне от 0 до 1. Высокое значение означает большую зависимость от последних градиентов.
                    epsilon=epsilon,                        # малый константный коэффициент, который предотвращает деление на ноль. Небольшое положительное значение, предотвращающее числовую нестабильность.
                    item_alpha=item_alpha,                  # регуляризационный параметр для элементов, который контролирует сложность модели. Большое значение уменьшает сложность модели, но может ухудшить производительность.
                    user_alpha=user_alpha,                  # аналогичен item_alpha, но применяется к пользователям. Также контролирует сложность модели, уменьшая ее при больших значениях.
                    max_sampled=max_sampled,                # максимум отрицательных примеров, которые будут использованы для каждого пользователя при обучении. Ограничивает количество отрицательных примеров, чтобы предотвратить слишком долгую тренировку.
                    random_state=random_state)              # устанавливаем случайное состояние для воспроизводимости результатов.
    
    model.fit(train, 
              epochs=10,
              )

    # Оцениваем модель
    p_at_k = precision_at_k(model, train, test, k=3)
    
    return p_at_k.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_params = study.best_trial.params
print("Лучшие параметры:", best_params)

[I 2025-01-25 17:39:43,424] A new study created in memory with name: no-name-f0e561f2-edee-4a12-b4dc-87969421a0bc


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2025-01-25 17:49:56,043] Trial 0 finished with value: 0.003028332954272628 and parameters: {'no_components': 71, 'k': 5, 'n': 33, 'loss': 'warp-kos', 'learning_schedule': 'adagrad', 'learning_rate': 4.7730294686031584e-05, 'rho': 0.06524675197574412, 'epsilon': 3.3648560079282e-06, 'item_alpha': 1.219725883878163e-05, 'user_alpha': 2.7125261982680836e-06, 'max_sampled': 52}. Best is trial 0 with value: 0.003028332954272628.
[I 2025-01-25 18:00:20,255] Trial 1 finished with value: 0.003637513844296336 and parameters: {'no_components': 77, 'k': 5, 'n': 14, 'loss': 'warp-kos', 'learning_schedule': 'adagrad', 'learning_rate': 0.0005606672694950152, 'rho': 0.11737832371983944, 'epsilon': 0.0002524539852528895, 'item_alpha': 0.00019031198208465836, 'user_alpha': 1.05511977715931e-06, 'max_sampled': 16}. Best is trial 1 with value: 0.003637513844296336.
[I 2025-01-25 18:07:37,133] Trial 2 finished with value: 0.0012535072164610028 and parameters: {'no_components': 41, 'k': 3, 'n': 49, 'los

In [29]:
# Визуализация истории хода оптимизации
optuna.visualization.plot_optimization_history(study, target_name="p_at_k")

In [30]:
# Визуализация важности гиперпараметров
optuna.visualization.plot_param_importances(study, target_name="p_at_k")

In [31]:
# Визуализация контуров гиперпараметров
optuna.visualization.plot_contour(study, params=["rho", "epsilon"], target_name="p_at_k")

In [32]:
# Визуализация среза
optuna.visualization.plot_slice(study, target_name="p_at_k")

In [33]:
# Визуализация прогресса оптимизации для параллельной координатной плоскости
optuna.visualization.plot_parallel_coordinate(study, target_name="p_at_k")

In [34]:
# Визуализация соотношения гиперпараметров к целевой метрике
optuna.visualization.plot_edf(study, target_name="p_at_k")

In [20]:
# Фиксируем время начала работы
start_time = time.time()
# Формируем параметры модели и обучаем её с подобранными параметрами
model = LightFM(**best_params,
                random_state=RANDOM_SEED,
                )
model.fit(train, epochs=200)

# Расчитываем метрику precision@k
train_precision = precision_at_k(model, train, k=3).mean()
test_precision = precision_at_k(model, test, k=3).mean()

# Фиксируем время окончания работы
end_time = time.time()
elapsed_time = end_time - start_time

print("Precision@3: train %.4f, test %.4f." % (train_precision, test_precision))
print()

# # Вариан 1 - Выбираем случайного пользователя
# random_normalized_visitorid = np.random.choice(users)

# Вариант 2 - Вводим оригинальный visitorid
original_visitorid = 890980
random_normalized_visitorid = original_visitorid - min_visitorid # приводим его к нормализованному виду

userid = user_to_idx[random_normalized_visitorid]

# Получаем рекомендации для пользователя
n_users, n_items = matrix.shape
scores = model.predict(userid, np.arange(n_items), num_threads=8)
top_items = np.argsort(-scores)[:3]  # берем топ-3 элемента

# Преобразуем обратно в оригинальные itemid
original_item_ids = [items[idx] + min_itemid for idx in top_items]

# Преобразуем обратно в оригинальный visitorid
original_visitorid = random_normalized_visitorid + min_visitorid

# Используем pandas для отображения результатов в виде таблицы
rec_user = pd.DataFrame({
    "visitorid": original_visitorid,
    "itemid": original_item_ids,
    "score": scores[top_items],
    "already_liked": np.in1d(top_items, matrix[userid].indices.ravel())  # преобразуем в одномерный массив
})

print(rec_user)
print()

set1 = set(df[df["visitorid"] == original_visitorid]["itemid"].unique().tolist())
set2 = set(original_item_ids)
print(f'Соответствие itemid между рекомендациями и фактом: {set1 & set2}')
print()

print(rec_user["already_liked"].value_counts())

# Сохраняем результаты
results = add_results(results, "LightFM_light_count_Optuna", elapsed_time, train_precision, test_precision)
print(results)

Precision@3: train 0.8319, test 0.0069.

   visitorid  itemid       score  already_liked
0     890980  364950 -827.996277           True
1     890980  230616 -830.303650           True
2     890980  355994 -830.376282           True

Соответствие itemid между рекомендациями и фактом: {230616, 355994, 364950}

already_liked
True    3
Name: count, dtype: int64
                          Model         Time train_Precision@3  \
0       LightFM_light_count_bpr   365.955931           0.00017   
1  LightFM_light_count_logistic   344.086594          0.004212   
2      LightFM_light_count_warp   365.557477          0.004288   
3  LightFM_light_count_warp-kos    384.49376          0.002981   
4                LightFM_hybrid  2046.534966          0.027237   
5    LightFM_light_count_Optuna  1723.582243          0.831884   

  test_Precision@3  
0         0.000018  
1         0.001503  
2         0.001439  
3         0.000947  
4         0.000892  
5         0.006857  


In [27]:
# Cохраняем модель в файл
dump(model, "model/model_LightFM_light_count_Optuna.joblib")
# Cохраняем результаты
results.to_csv("results/results_lightfm_light_count.csv", sep=",", index=False)

In [36]:
best_params

{'no_components': 85,
 'k': 1,
 'n': 39,
 'loss': 'warp',
 'learning_schedule': 'adadelta',
 'learning_rate': 0.00019915658230327475,
 'rho': 0.1975069513891302,
 'epsilon': 0.0007031225068678744,
 'item_alpha': 3.775802090847933e-05,
 'user_alpha': 4.277180404639178e-06,
 'max_sampled': 34}

In [37]:
# Фиксируем время начала работы
start_time = time.time()

# Формируем параметры модели
model_hybrid = LightFM(no_components=85,
                       k=1,
                       loss="warp",
                       learning_schedule="adadelta",
                       learning_rate=0.0002,
                       rho=0.1975,
                       epsilon=7e-4,
                       max_sampled=34,
                       random_state=RANDOM_SEED,
                       )

# Обучение гибридной модели. Обратите внимание.
# Обратите внимание, что на этот раз мы проходим по элементам в матрице элементов.
model_hybrid = model_hybrid.fit(train,
                item_features=item_features,
                epochs=200,
                )

# Оценим метрику precision@k
train_precision = precision_at_k(model_hybrid, train, item_features=item_features, k=3).mean()
test_precision = precision_at_k(model_hybrid, test, item_features=item_features, k=3).mean()

# Фиксируем время окончания работы
end_time = time.time()
elapsed_time = end_time - start_time

print("Precision@3: train %.4f, test %.4f." % (train_precision, test_precision))
print()

# # Вариан 1 - Выбираем случайного пользователя
# random_normalized_visitorid = np.random.choice(users)

# Вариант 2 - Вводим оригинальный visitorid
original_visitorid = 890980
random_normalized_visitorid = original_visitorid - min_visitorid # приводим его к нормализованному виду

userid = user_to_idx[random_normalized_visitorid]

# Получаем рекомендации для пользователя
n_users, n_items = matrix.shape
scores = model_hybrid.predict(userid, np.arange(n_items), num_threads=8)
top_items = np.argsort(-scores)[:3]  # берем топ-3 элемента

# Преобразуем обратно в оригинальные itemid
original_item_ids = [items[idx] + min_itemid for idx in top_items]

# Преобразуем обратно в оригинальный visitorid
original_visitorid = random_normalized_visitorid + min_visitorid

# Используем pandas для отображения результатов в виде таблицы
rec_user = pd.DataFrame({
    "visitorid": original_visitorid,
    "itemid": original_item_ids,
    "score": scores[top_items],
    "already_liked": np.in1d(top_items, matrix[userid].indices.ravel())  # преобразуем в одномерный массив
})

print(rec_user)
print()

set1 = set(df[df["visitorid"] == original_visitorid]["itemid"].unique().tolist())
set2 = set(original_item_ids)
print(f'Соответствие itemid между рекомендациями и фактом: {set1 & set2}')
print()

print(rec_user["already_liked"].value_counts())

# Сохраняем результаты
results = add_results(results, "LightFM_light_count_hybrid_Optuna", elapsed_time, train_precision, test_precision)
print(results)

Precision@3: train 0.6125, test 0.0017.

   visitorid  itemid        score  already_liked
0     890980  283399 -8642.464844           True
1     890980  236302 -8787.856445           True
2     890980  407992 -8787.876953           True

Соответствие itemid между рекомендациями и фактом: {407992, 236302, 283399}

already_liked
True    3
Name: count, dtype: int64
                               Model         Time train_Precision@3  \
0            LightFM_light_count_bpr   365.955931           0.00017   
1       LightFM_light_count_logistic   344.086594          0.004212   
2           LightFM_light_count_warp   365.557477          0.004288   
3       LightFM_light_count_warp-kos    384.49376          0.002981   
4                     LightFM_hybrid  2046.534966          0.027237   
5         LightFM_light_count_Optuna  1723.582243          0.831884   
6  LightFM_light_count_hybrid_Optuna  5063.707717          0.612461   

  test_Precision@3  
0         0.000018  
1         0.001503  
2  

In [38]:
# Cохраняем модель в файл
dump(model, "model/model_LightFM_light_count_hybrid_Optuna.joblib")
# Cохраняем результаты
results.to_csv("results/results_lightfm_light_count.csv", sep=",", index=False)